# End-to-End Machine Learning Pipeline with Amazon SageMaker

**Project Overview:** In this notebook, we are building a Machine Learning system to predict the progression of diabetes in patients. We will use historical health data to train a machine learning algorithm, determine the best settings for it, and deploy it as a live web service (API) that can make real-time predictions.

## Phase 1: Environment Setup and Data Preparation
We use a well-known built-in sklearn medical dataset containing health metrics for diabetes patients—such as Age, BMI, and average blood pressure—alongside a target value indicating how their disease actually progressed.

We use the Python library `boto3` (the official AWS SDK) to request our IAM Role and Default S3 Bucket from AWS. Machine learning on SageMaker requires decoupled storage. Training instances are ephemeral and do not store data locally long-term. We define an S3 bucket and prefix path that will act as the central repository for our datasets, logs, and compiled model artifacts.

In [1]:
import sagemaker
import boto3
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = sagemaker_session.default_bucket()
prefix = "sagemaker/end-to-end-ml-pipeline"

print(f"SageMaker Role: {role}")
print(f"Default Bucket: s3://{bucket}")
print(f"Prefix: {prefix}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
SageMaker Role: arn:aws:iam::406682760260:role/service-role/AmazonSageMaker-ExecutionRole-20260217T223000
Default Bucket: s3://sagemaker-ap-southeast-2-406682760260
Prefix: sagemaker/end-to-end-ml-pipeline


We partition the diabetes dataset into three sets: Training Set (70%), Validation Set (15%), and Testing Set (15%). After splitting, we convert the data structures into CSV format and upload them to Amazon S3. SageMaker training jobs will pull data directly from these S3 URIs rather than your local notebook memory.

In [2]:
# Load and prepare data
data = load_diabetes()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.DataFrame(data.target, columns=['target'])

# Split into train/validation/test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

# Combine features and targets
train_data = pd.concat([y_train, X_train], axis=1)
val_data = pd.concat([y_val, X_val], axis=1)
test_data = pd.concat([y_test, X_test], axis=1)

# Save locally
train_data.to_csv('train.csv', index=False, header=False)
val_data.to_csv('val.csv', index=False, header=False)
test_data.to_csv('test.csv', index=False, header=False)

# Upload to S3
train_s3_uri = sagemaker_session.upload_data(
    'train.csv', bucket=bucket, key_prefix=f'{prefix}/data/train'
)
val_s3_uri = sagemaker_session.upload_data(
    'val.csv', bucket=bucket, key_prefix=f'{prefix}/data/val'
)
test_s3_uri = sagemaker_session.upload_data(
    'test.csv', bucket=bucket, key_prefix=f'{prefix}/data/test'
)

print(f"Training data uploaded to: {train_s3_uri}")

Training data uploaded to: s3://sagemaker-ap-southeast-2-406682760260/sagemaker/end-to-end-ml-pipeline/data/train/train.csv


## Phase 2: Model Training with XGBoost

We initialize an Estimator, which is SageMaker's high-level interface for training models. In this example, we select XGBoost (Extreme Gradient Boosting) for the model, an optimized, distributed gradient-boosting library highly efficient for tabular data. Then, we define the algorithm's Hyperparameters:
- `max_depth` = The maximum depth of a tree. The larger the value, the more complex the tree can become, capturing more detailed patterns, but with a higher risk of overfitting. A value of 5 is quite moderate and is typically used to control model complexity.
- `eta` = The learning rate or step size shrinkage. It's often also called learning_rate. It controls how much each new tree contributes to the model. Small values (e.g., 0.01–0.3) make the model more robust but require more trees. 0.2 is a common value that converges reasonably fast.
- `gamma` = The minimum loss reduction required to make a further partition (split) on a leaf node. The larger the gamma value, the more conservative the model (fewer splits). A value of 4 is relatively large, so the model tends to avoid splits that don't significantly reduce the loss. This helps reduce overfitting.
- `min_child_weight` = The minimum sum of instance weight needed in a child node. A larger value prevents the creation of child nodes that are too small, leading to a simpler model. A value of 6 means that if the sample weight is less than 6, a split will not be performed. This also helps control overfitting.
- `subsample` = The fraction of data to be randomly sampled for building each tree. A value of 0.8 means 80% of the data is used for each tree. This is similar to bagging and helps reduce overfitting while improving generalization.
- `objective` = The learning task and the corresponding loss function to be used. `reg:squarederror` means the model performs regression with the mean squared error (MSE) loss. It is suitable for predicting continuous values (regression problems).
- `num_round` = The number of boosting rounds or the number of trees to be built (commonly referred to as n_estimators). 100 trees are a fairly common number. More trees can improve performance, but also increase computation time and the risk of overfitting if not balanced with regularization.
- `verbosity` = The level of detail for messages printed during training. A value of 1 prints standard informational messages. 0 prints nothing, and 2 prints more detailed messages.

After we finish initializing the Estimator, we can execute the model with `.fit()`. Executing the `.fit()` method triggers the following infrastructure workflow:
1. SageMaker provisions a dedicated, ephemeral compute instance (`ml.m5.large`).
2. It pulls the pre-built XGBoost Docker container from Amazon ECR.
3. It downloads the training and validation CSV files from S3 into the instance.
4. It executes the training script until convergence or the defined epoch limit.
5. It compresses the finalized model weights into a `model.tar.gz` artifact, uploads it to S3, and terminates the compute instance to halt billing.

In [3]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput

# Get XGBoost container
container = sagemaker.image_uris.retrieve(
    framework='xgboost',
    region=boto3.Session().region_name,
    version='1.5-1'
)

# Configure hyperparameters
hyperparameters = {
    'max_depth': '5',
    'eta': '0.2',
    'gamma': '4',
    'min_child_weight': '6',
    'subsample': '0.8',
    'objective': 'reg:squarederror',
    'num_round': '100',
    'verbosity': '1'
}

# Create estimator
xgb_estimator = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type='ml.m5.large',
    volume_size=30,
    output_path=f's3://{bucket}/{prefix}/models',
    hyperparameters=hyperparameters,
    sagemaker_session=sagemaker_session
)

# Define data channels
train_input = TrainingInput(train_s3_uri, content_type='text/csv')
val_input = TrainingInput(val_s3_uri, content_type='text/csv')

# Launch training job
xgb_estimator.fit(
    {'train': train_input, 'validation': val_input},
    logs=True
)

INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-02-19-03-43-32-485


2026-02-19 03:43:36 Starting - Starting the training job...
2026-02-19 03:43:49 Starting - Preparing the instances for training...
2026-02-19 03:44:11 Downloading - Downloading input data...
2026-02-19 03:44:51 Downloading - Downloading the training image......
2026-02-19 03:46:02 Training - Training image download completed. Training in progress.
2026-02-19 03:46:02 Uploading - Uploading generated training model/miniconda3/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
[2026-02-19 03:45:55.597 ip-10-0-165-11.ap-southeast-2.compute.internal:6 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-02-19 03:45:55.634 ip-10-0-165-11.ap-southeast-2.compute.internal:6 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-02-19:03:45:56:INFO] Imported framework sagemaker_xgb

Machine learning training is often opaque. SageMaker Debugger allows us to capture internal state data (tensors) and metrics at regular intervals during the training job.

By configuring the `DebuggerHookConfig`, we instruct the training container to emit specific data points—such as validation loss and feature importance—every 10 steps. This data is saved directly to S3, enabling us to track convergence, detect issues such as vanishing gradients or overfitting, and visualize the model's learning curve in real time.

In [4]:
from sagemaker.debugger import DebuggerHookConfig, CollectionConfig

# Configure Debugger
debugger_config = DebuggerHookConfig(
    s3_output_path=f's3://{bucket}/{prefix}/debugger',
    collection_configs=[
        CollectionConfig(
            name='metrics',
            parameters={'save_interval': '10'}
        ),
        CollectionConfig(
            name='feature_importance',
            parameters={'save_interval': '10'}
        )
    ]
)

# Update estimator with Debugger
xgb_estimator_debug = Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type='ml.m5.large',
    output_path=f's3://{bucket}/{prefix}/models',
    hyperparameters=hyperparameters,
    debugger_hook_config=debugger_config,  # Enable Debugger
    sagemaker_session=sagemaker_session
)

# Run training with debugging
xgb_estimator_debug.fit({'train': train_input, 'validation': val_input})

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: sagemaker-xgboost-2026-02-19-03-46-49-447


2026-02-19 03:46:51 Starting - Starting the training job...
2026-02-19 03:47:05 Starting - Preparing the instances for training...
2026-02-19 03:47:25 Downloading - Downloading input data...
2026-02-19 03:48:11 Downloading - Downloading the training image......
2026-02-19 03:49:22 Training - Training image download completed. Training in progress.
2026-02-19 03:49:22 Uploading - Uploading generated training model/miniconda3/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index
[2026-02-19 03:49:16.046 ip-10-0-199-148.ap-southeast-2.compute.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-02-19 03:49:16.071 ip-10-0-199-148.ap-southeast-2.compute.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-02-19:03:49:16:INFO] Imported framework sagemaker_x

## Phase 3: Hyperparameter Optimization (HPO)

Hard-coding hyperparameters rarely yields the optimal model. Instead, we use a HyperparameterTuner to automate the search for the best configuration. Rather than passing specific values, we define a search space (e.g., a continuous range for the learning rate `eta`). 

SageMaker orchestrates a series of parallel training jobs. Using Bayesian Optimization, it evaluates the objective metric (e.g., validation RMSE) from previous runs to intelligently select the hyperparameter combinations for subsequent runs. Once the maximum number of jobs is reached, it identifies the model artifact that achieved the lowest error rate.

In [5]:
from sagemaker.tuner import (
    IntegerParameter,
    ContinuousParameter,
    HyperparameterTuner
)

# Define hyperparameter ranges
hyperparameter_ranges = {
    'max_depth': IntegerParameter(3, 10),
    'eta': ContinuousParameter(0.05, 0.5),
    'min_child_weight': IntegerParameter(1, 10),
    'subsample': ContinuousParameter(0.5, 1.0)
}

# Create tuner
tuner = HyperparameterTuner(
    estimator=xgb_estimator,
    objective_metric_name='validation:rmse',
    objective_type='Minimize',
    hyperparameter_ranges=hyperparameter_ranges,
    max_jobs=10,
    max_parallel_jobs=2,
    base_tuning_job_name='xgboost-tuning'
)

# Launch tuning job
tuner.fit({'train': train_input, 'validation': val_input})

# Get the best training job
best_training_job = tuner.best_training_job()
print(f"Best training job: {best_training_job}")

INFO:sagemaker:Creating hyperparameter tuning job with name: xgboost-tuning-260219-0350


........................................................................................!
Best training job: xgboost-tuning-260219-0350-010-eb71c671


## Phase 4: Model Deployment and Inference

The optimized model currently exists as a static `.tar.gz` artifact in S3. To perform inference, it must be loaded into memory on a persistent server. Calling `.deploy()` provisions a permanent EC2 instance (`ml.m5.large`), loads the XGBoost inference container and our model artifact, and exposes it via a secure REST API endpoint. 

We also configure Serializers and Deserializers. The HTTP endpoint operates on byte streams and expects standard payload formats like CSV or JSON. The `CSVSerializer` formats our native Python arrays into a comma-separated string for the HTTP POST request, while the `CSVDeserializer` parses the HTTP response payload back into a Python float for our notebook to use.

In [6]:
import boto3
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import CSVDeserializer

sm_client = boto3.client('sagemaker')
endpoint_name = 'diabetes-predictor'

# 1. Automatically delete the existing endpoint if it exists
try:
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print(f"Deleted existing endpoint: {endpoint_name}")
except Exception:
    pass # Endpoint doesn't exist yet

# 2. Automatically delete the existing endpoint configuration if it exists
try:
    sm_client.delete_endpoint_config(EndpointConfigName=endpoint_name)
    print(f"Deleted existing endpoint configuration: {endpoint_name}")
except Exception:
    pass # Config doesn't exist yet

print("Deploying new endpoint (this will take 3-5 minutes)...")

# 3. Deploy the model
best_model = sagemaker.estimator.Estimator.attach(best_training_job)

predictor = best_model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',
    endpoint_name=endpoint_name
)

# Attach serializers (to prevent the previous List/CSV error!)
predictor.serializer = CSVSerializer()
predictor.deserializer = CSVDeserializer()

# 4. Test the endpoint
test_sample = test_data.iloc[:5, 1:].values.tolist()
predictions = predictor.predict(test_sample)
print(f"Sample predictions: {predictions}")

Deleted existing endpoint: diabetes-predictor
Deleted existing endpoint configuration: diabetes-predictor
Deploying new endpoint (this will take 3-5 minutes)...

2026-02-19 03:57:33 Starting - Found matching resource for reuse
2026-02-19 03:57:33 Downloading - Downloading the training image
2026-02-19 03:57:33 Training - Training image download completed. Training in progress.
2026-02-19 03:57:33 Uploading - Uploading generated training model
2026-02-19 03:57:33 Completed - Resource retained for reuse

INFO:sagemaker:Creating model with name: xgboost-tuning-2026-02-19-03-57-47-766


INFO:sagemaker:Creating endpoint-config with name diabetes-predictor
INFO:sagemaker:Creating endpoint with name diabetes-predictor


------!Sample predictions: [['87.66477966308594'], ['114.79295349121094'], ['106.43512725830078'], ['175.5408172607422'], ['86.56935119628906']]


In production environments, data distributions can shift over time (data drift or concept drift), degrading model accuracy. Enabling Data Capture configures the endpoint to automatically sample and record incoming HTTP request payloads (features) and outgoing responses (predictions). These logs are serialized and stored in S3. Later, services like SageMaker Model Monitor can analyze these logs against the baseline dataset established during training to detect statistical drift and trigger retraining pipelines.

In [7]:
from sagemaker.model_monitor import DataCaptureConfig

# Update endpoint with data capture
predictor.update_data_capture_config(
    DataCaptureConfig(
        enable_capture=True,
        sampling_percentage=50,  # Capture 50% of requests
        destination_s3_uri=f's3://{bucket}/{prefix}/monitoring/data-capture'
    )
)

INFO:sagemaker:Creating endpoint-config with name diabetes-predictor-2026-02-19-04-01-20-307


------!

## Phase 5: Real-Time Inference Execution

With the endpoint active, we evaluate the model against the holdout Testing Set established in Phase 1.  We iterate through the testing feature vectors, passing each via the `.predict()` method. The SageMaker SDK handles the serialization, network request to the deployed REST API, and deserialization of the response. The endpoint processes the features through the in-memory XGBoost model and returns the predicted severity score in milliseconds, validating the end-to-end functionality of the pipeline.

In [8]:
import random
import time

print("--- 1. Testing Real-time Inference ---")

# Take the first 20 rows of test data
test_features = test_data.iloc[:20, 1:].values.tolist()

for i, features in enumerate(test_features):
    # Send the data to the endpoint
    # (The CSVSerializer we added earlier handles converting this list to bytes)
    prediction = predictor.predict([features])
    
    # Print the first 3 features of the row and the resulting prediction
    print(f"Request {i+1} | Input: {features[:3]}... -> Prediction: {prediction}")
    
    # Add a small delay to simulate real-world traffic
    time.sleep(random.uniform(0.1, 0.5))

print("End-to-End Inference Pipeline Test Complete!")

--- 1. Testing Real-time Inference ---
Request 1 | Input: [-0.09632801625429555, -0.044641636506989144, -0.06979686649477428]... -> Prediction: [['87.66477966308594']]
Request 2 | Input: [-0.06726770864614018, 0.05068011873981862, -0.012672826579091896]... -> Prediction: [['114.79295349121094']]
Request 3 | Input: [-0.10359309315633439, -0.044641636506989144, -0.03746250427835029]... -> Prediction: [['106.43512725830078']]
Request 4 | Input: [0.04170844488444244, 0.05068011873981862, -0.022373135244019075]... -> Prediction: [['175.5408172607422']]
Request 5 | Input: [-0.074532785548179, 0.05068011873981862, -0.00943939035744949]... -> Prediction: [['86.56935119628906']]
Request 6 | Input: [-0.045472477940023646, -0.044641636506989144, 0.039062152967186486]... -> Prediction: [['179.3427276611328']]
Request 7 | Input: [0.02354575262934534, 0.05068011873981862, -0.03746250427835029]... -> Prediction: [['133.24256896972656']]
Request 8 | Input: [-0.02367724723390713, -0.044641636506989144,